# **Computer Vision with Deep Learning**

## Visual Recognition

## Lab2 - Optional - Image classification with CNNs

2021 - Veronica Vilaplana - [GPI @ IDEAI](https://imatge.upc.edu/web/) Research group // [ETSETB – UPC.TelecosBCN](https://telecos.upc.edu/ca)



The goal of this assignment is to get hands on experience designing and trainind deep convolutional networks for image classification using PyTorch.
In this assignment you will use PyTorch, which is currently one of the most popular deep learning frameworks.


This lab is adapted from the Pytorch tutorial https://colab.research.google.com/github/pytorch/tutorials/blob/gh-pages/_downloads/cifar10_tutorial.ipynb

We will train and test a simple CNN to classify images from the CIFAR10 dataset.

<font color=red>Advice: Select the GPU Hardware Acceleration in the Runtime environment Menu to train the network fast.</font>


We will use a package called
``torchvision``, that consists of popular datasets, model architectures, and common image transformations for computer vision.

In this lab will use the CIFAR10 dataset.
It has 10 classes: ‘airplane’, ‘automobile’, ‘bird’, ‘cat’, ‘deer’, ‘dog’, ‘frog’, ‘horse’, ‘ship’, ‘truck’. The images in CIFAR-10 are of size 3x32x32, i.e. 3-channel color images of 32x32 pixels in size.

We will learn how to:

1. Load and normalize the CIFAR10 training and test datasets using ``torchvision``
2. Define a Convolutional Neural Network (BaseNet)
3. Define the loss function
4. Train the network on the training data
5. Test the network on the test data

In the first part of this notebook code is provided to solve the task using a simple baseline CNN.

Then, you will have to complete some tasks trying to improve the model.

Instructions:
1. You can work in teams of two.
2. Add and run your Python code and answer the questions in this same notebook.
3. Upload a zip file with the notebook (after running code cells, including results, confussion matrices, images, etc.) to Atenea.

### 1. Loading and normalizing CIFAR10

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
# Assume that we are on a CUDA machine, then this should print a CUDA device:
print(device)


Defining the PyTorch Dataset and the DataLoader:

The PyTorch Dataset (https://pytorch.org/docs/stable/data.html#torch.utils.data.Dataset) is an inheritable `class` that helps us defining what source of data do we have (image, audio, text, ...) and how to load it. The CIFAR dataset is easible accessible from it.

The output of torchvision datasets are PILImage images of range [0, 1]. We will transform them to Tensors of normalized range [-1, 1].


In [ ]:
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=4,
                                          shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=4,
                                         shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat',
           'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

We show some images


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

# functions to show an image
def imshow(img):
    img = img / 2 + 0.5     # unnormalize
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))

# get some random training images from the dataloader by running over its iterator
dataiter = iter(trainloader)
images, labels = next(dataiter)

# show images
imshow(torchvision.utils.make_grid(images))
# print labels
print(' '.join('%5s' % classes[labels[j]] for j in range(4)))

### 2. Define a Convolutional Neural Network
We define a neural network where the inputs are 3-channel images

This is a basic network that you should understand, run and eventually improve (see Tasks at the end).

You may want to add more conv layers, add more fully connected layers or add regularization layers like Batchnorm.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class BaseNet(nn.Module):
    def __init__(self):
        super(BaseNet, self).__init__()

        # <<TODO>> Add more conv layers with increasing
        # output channels
        # <<TODO>> Add normalization layers after conv
        # layers (nn.BatchNorm2d)

        # Also experiment with kernel size in conv2d layers (say 3
        # inspired from VGGNet)
        # To keep it simple, keep the same kernel size
        # (right now set to 5) in all conv layers.
        # Do not have a maxpool layer after every conv layer in your
        # deeper network as it leads to too much loss of information.

        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):

        # <<TODO>> Based on the above edits, you'll have
        # to edit the forward pass description here.

        x = self.pool(F.relu(self.conv1(x)))
        # Output size = 28//2 x 28//2 = 14 x 14

        x = self.pool(F.relu(self.conv2(x)))
        # Output size = 10//2 x 10//2 = 5 x 5

        x = x.view(-1, 16 * 5 * 5)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)

        # No softmax is needed as the loss function in step 3
        # takes care of that

        return x

net = BaseNet()

# For training on GPU, we need to transfer net and data onto the GPU
# http://pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html#training-on-gpu
net.to(device)


### 3. Define a Loss function and optimizer

We will use the Cross-Entropy loss and SGD with momentum. The CrossEntropyLoss criterion already includes softmax within its
implementation. That's why we don't use a softmax in our model definition.

We set hyperparameters: learning rate = 0.001 and momentum = 0.9. Later you can tune the learning rate and see whether the momentum is useful or not


In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)


### 4. Train the network

To train the network we loop over the data iterator feed the inputs to the network and optimize.

We evaluate the validation accuracy at each
epoch and plot these values over the number of epochs

Nothing to change here

We will first train the network for 2 epochs



In [ ]:
for epoch in range(2):  # loop over the dataset multiple times

    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        # get the inputs
        inputs, labels = data
        inputs, labels = inputs.to(device), labels.to(device)

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # print statistics
        running_loss += loss.item()
        if i % 2000 == 1999:    # print every 2000 mini-batches
            print('[%d, %5d] loss: %.3f' %
                  (epoch + 1, i + 1, running_loss / 2000))
            running_loss = 0.0


print('Finished Training')

### 5. Test the network on the test data

We will evaluate the performance on the test set.

First, let's see the predictions for a few samples.



In [ ]:
dataiter = iter(testloader)
images, labels = next(dataiter)

# print images
imshow(torchvision.utils.make_grid(images))
print('GroundTruth: ', ' '.join('%5s' % classes[labels[j]] for j in range(4)))

images, labels = images.to(device), labels.to(device)



In [ ]:
outputs = net(images)
_, predicted = torch.max(outputs, 1)

print('Predicted: ', ' '.join('%5s' % classes[predicted[j]]
                              for j in range(4)))

Now we compute the performance on the whole dataset.

In [ ]:
# Check out why .eval() is important!
# https://discuss.pytorch.org/t/model-train-and-model-eval-vs-model-and-model-eval/5744/2
net.eval()

correct = 0
total = 0
#with torch.no_grad():
for data in testloader:
    images, labels = data
    images, labels = images.to(device), labels.to(device)
    outputs = net(images)
    _, predicted = torch.max(outputs.data, 1)
    total += labels.size(0)
    correct += (predicted == labels).sum().item()

print('Accuracy of the network on the 10000 test images: %d %%' % (
    100 * correct / total))

This accuracy is much better than chance, which is 10% accuracy (randomly picking
a class out of 10 classes).

Next we compute the accuracy per class

In [ ]:
class_correct = list(0. for i in range(10))
class_total = list(0. for i in range(10))
with torch.no_grad():
    for data in testloader:
        images, labels = data
        images, labels = images.to(device), labels.to(device)
        outputs = net(images)
        _, predicted = torch.max(outputs, 1)

        c = (predicted == labels).squeeze()
        for i in range(4):
            label = labels[i]
            class_correct[label] += c[i].item()
            class_total[label] += 1


for i in range(10):
    print('Accuracy of %5s : %2d %%' % (
        classes[i], 100 * class_correct[i] / class_total[i]))

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sn
import pandas as pd

y_pred = []
y_true = []
with torch.no_grad():
    for images, labels in testloader:

        images, labels = images.to(device), labels.to(device)

        output = net(images)
        output = (torch.max(output, 1)[1]).data.cpu().numpy()
        y_pred.extend(output) #save prediction

        labels =labels.data.cpu().numpy()
        y_true.extend(labels)   #save truth

cm = confusion_matrix(y_true,y_pred)
print(cm)


In [ ]:
df_cm = pd.DataFrame(cm/np.sum(cm) *10, index = [i for i in classes],
                     columns = [i for i in classes])
plt.figure(figsize = (12,7))
sn.heatmap(df_cm, annot=True)
plt.show()

In [ ]:
print(len(y_true))
print(len(y_pred))

##Tasks

Try to improve the baseline architecture provided.
Edit the BaseNet class or make new classes for devising a more accurate deep net architecture. In your report include a tabla describing the layers of the final network.

For improving the network you should consider (some of) the following:

1. Add more conv layers with increasing output channels
2. Add normalization layers after conv layers (nn.BatchNorm2d)
3. Also experiment with kernel size in conv2d layers (say 3, inspired from VGGNet) (but for simplicity  use always the same size)
4. Add more linear (fc) layers

Note:
*   Do not add a maxpool layer after every conv layer in your deeper networks as it leads to too much loss of information (orignial cifar images are very small!).

<font color=blue>Answer: Here explain your experiments, compare and comment results.</font>